In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import cv2
import numpy as np
from glob import glob

# ========= PATH =========
input_dir = "/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset/100468003_L.png"
output_dir = "/kaggle/working/vessel_method1"
os.makedirs(output_dir, exist_ok=True)

image_paths = glob(os.path.join(input_dir, "*.png")) + glob(os.path.join(input_dir, "*.jpg")) + glob(os.path.join(input_dir, "*.jpeg"))

for img_path in image_paths:
    img = cv2.imread(img_path)
    if img is None:
        continue

    # Green channel extract
    green = img[:, :, 1]

    # CLAHE contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    green_clahe = clahe.apply(green)

    # Blackhat to highlight dark vessels
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    blackhat = cv2.morphologyEx(green_clahe, cv2.MORPH_BLACKHAT, kernel)

    # Normalize
    blackhat = cv2.normalize(blackhat, None, 0, 255, cv2.NORM_MINMAX)

    # Threshold
    _, vessel = cv2.threshold(blackhat, 25, 255, cv2.THRESH_BINARY)

    # Background white, vessel black
    vessel_map = 255 - vessel

    # Save
    name = os.path.basename(img_path)
    save_path = os.path.join(output_dir, name)
    cv2.imwrite(save_path, vessel_map)

print("Method 1 done. Saved to:", output_dir)

In [ ]:
import cv2
import os
import numpy as np
from glob import glob
from skimage.filters import frangi

In [ ]:
input_folder = "/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset/100468003_L.png"
output_folder = "/kaggle/working/vessel_method1/"
os.makedirs(output_folder, exist_ok=True)

image_paths = glob(input_folder + "*.png") + glob(input_folder + "*.jpg") + glob(input_folder + "*.jpeg")
print("Total images:", len(image_paths))

In [ ]:
def vessel_method1(img):
    green = img[:, :, 1]

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(green)

    norm = enhanced.astype(np.float32) / 255.0
    vesselness = frangi(norm)

    vesselness = (vesselness - vesselness.min()) / (vesselness.max() - vesselness.min() + 1e-8)
    vesselness = (vesselness * 255).astype(np.uint8)

    _, mask = cv2.threshold(vesselness, 30, 255, cv2.THRESH_BINARY)

    white_bg = np.ones_like(mask) * 255
    white_bg[mask > 0] = 0

    return white_bg

In [ ]:
for path in image_paths:
    img = cv2.imread(path)
    if img is None:
        print("Failed:", path)
        continue

    name = os.path.basename(path).rsplit(".", 1)[0]
    vessel_map = vessel_method1(img)

    save_path = os.path.join(output_folder, f"{name}_vessel_m1.png")
    cv2.imwrite(save_path, vessel_map)

print("Method 1 done!")

In [ ]:
import shutil

zip_path = "/kaggle/working/vessel_method16.zip"
shutil.make_archive(zip_path.replace(".zip", ""), 'zip', output_folder)
print("ZIP saved:", zip_path)

In [ ]:
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt
from skimage.filters import frangi
from glob import glob

# Input folder
input_folder = "/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset/"
# Output folder
output_folder = "/kaggle/working/vessel_maps_white_bg/"
os.makedirs(output_folder, exist_ok=True)

# Select first 50 images
image_paths = sorted(glob(os.path.join(input_folder, "*.png")))[:50]
print("Total selected images:", len(image_paths))

def vessel_enhance_white_bg(image_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Normalize to 0-1
    gray_norm = gray / 255.0
    
    # Frangi vessel enhancement
    vessel = frangi(gray_norm)
    
    # scale to 0-255
    vessel = (vessel / vessel.max() * 255).astype(np.uint8)
    
    # Invert: vessels black (0), background white (255)
    vessel_invert = 255 - vessel
    
    return gray, vessel_invert

# Loop and process
for i, path in enumerate(image_paths):
    original, vessel_map = vessel_enhance_white_bg(path)
    
    # Save vessel map
    fname = os.path.basename(path)
    save_path = os.path.join(output_folder, fname.replace(".png","_vessel.png"))
    cv2.imwrite(save_path, vessel_map)
    
    # Plot original + vessel map
    plt.figure(figsize=(10,5))
    plt.subplot(1,2,1)
    plt.imshow(original[..., ::-1])
    plt.title("Original")
    plt.axis("off")
    
    plt.subplot(1,2,2)
    plt.imshow(vessel_map, cmap='gray')
    plt.title("Vessel Map (White Background)")
    plt.axis("off")
    
    plt.show()
    
    if i==4:  # only show first 5 for preview
        break

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from glob import glob
from skimage.filters import frangi

# Path to your images folder
image_folder = "/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset"
save_folder = "/kaggle/working/vessel_maps_frangi"
os.makedirs(save_folder, exist_ok=True)

# Get first 50 images
image_paths = glob(os.path.join(image_folder, "*.png"))[:50]

def vessel_enhance_frangi(image_path):
    img = cv2.imread(image_path)
    green = img[:,:,1]  # Use green channel

    # Normalize
    green = green / 255.0

    # Apply Frangi filter
    vessels = frangi(green)
    
    # Convert to 0-255 and invert: vessels black, background white
    vessels = (1 - vessels) * 255
    vessels = vessels.astype(np.uint8)

    return img, vessels

# Loop through images
for i, img_path in enumerate(image_paths):
    original, vessel = vessel_enhance_frangi(img_path)

    # Plot original and vessel enhanced
    plt.figure(figsize=(10,5))
    plt.subplot(1,2,1)
    plt.imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    plt.title("Original")
    plt.axis('off')

    plt.subplot(1,2,2)
    plt.imshow(vessel, cmap='gray')
    plt.title("Vessel Enhanced (Black Vessels)")
    plt.axis('off')

    plt.show()

    # Save vessel enhanced image
    base_name = os.path.basename(img_path)
    save_path = os.path.join(save_folder, base_name)
    cv2.imwrite(save_path, vessel)

print("Processing complete. Vessel maps saved at:", save_folder)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from glob import glob
from skimage.filters import frangi

image_folder = "/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset"
save_folder = "/kaggle/working/vessel_maps_frangi_clahe"
os.makedirs(save_folder, exist_ok=True)

image_paths = glob(os.path.join(image_folder, "*.png"))[:50]

def vessel_enhance_frangi_clahe(image_path):
    img = cv2.imread(image_path)
    green = img[:,:,1]  # green channel
    
    # CLAHE for contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    green_eq = clahe.apply(green)
    
    # Normalize to 0-1
    green_norm = green_eq / 255.0

    # Frangi filter
    vessels = frangi(green_norm)
    
    # Invert: vessels black, background white
    vessels = (1 - vessels) * 255
    vessels = vessels.astype(np.uint8)

    return img, vessels

for i, img_path in enumerate(image_paths):
    original, vessel = vessel_enhance_frangi_clahe(img_path)

    plt.figure(figsize=(10,5))
    plt.subplot(1,2,1)
    plt.imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    plt.title("Original")
    plt.axis('off')

    plt.subplot(1,2,2)
    plt.imshow(vessel, cmap='gray')
    plt.title("Vessel Enhanced (Black Vessels)")
    plt.axis('off')
    plt.show()

    base_name = os.path.basename(img_path)
    save_path = os.path.join(save_folder, base_name)
    cv2.imwrite(save_path, vessel)

print("Processing complete. Vessel maps saved at:", save_folder)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob

image_paths = glob.glob("/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset/*.png")[:50]

for path in image_paths:
    img = cv2.imread(path)
    green = img[:,:,1]
    
    # Contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(green)
    
    # Top-hat filtering
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15,15))
    tophat = cv2.morphologyEx(enhanced, cv2.MORPH_TOPHAT, kernel)
    
    # Invert for black vessels
    vessel_map = 255 - tophat
    vessel_map = cv2.normalize(vessel_map, None, 0, 255, cv2.NORM_MINMAX)
    
    # Plot
    plt.figure(figsize=(8,4))
    plt.subplot(1,2,1)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Original")
    plt.axis('off')
    
    plt.subplot(1,2,2)
    plt.imshow(vessel_map, cmap='gray')
    plt.title("CLAHE + TopHat")
    plt.axis('off')
    
    plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
from skimage.filters import frangi

# 1. Image path
image_folder = "/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset"
image_paths = glob.glob(os.path.join(image_folder, "*.png"))[:50]  # first 50 images

# 2. Output folder
output_folder = "/kaggle/working/zhang_vessel"
os.makedirs(output_folder, exist_ok=True)

# 3. Loop over images
for idx, path in enumerate(image_paths):
    img = cv2.imread(path)
    green = img[:,:,1]  # use green channel

    # 3a. Apply Zhang (Frangi vesselness) filter
    vessel_map = frangi(green, scale_range=(1,5), scale_step=1)

    # 3b. Normalize and invert to get black vessels
    vessel_map = cv2.normalize(vessel_map, None, 0, 255, cv2.NORM_MINMAX)
    vessel_map = 255 - vessel_map
    vessel_map = vessel_map.astype(np.uint8)

    # 3c. Save the enhanced image
    filename = os.path.basename(path)
    save_path = os.path.join(output_folder, filename)
    cv2.imwrite(save_path, vessel_map)

    # 3d. Plot original vs vessel map
    plt.figure(figsize=(8,4))
    plt.subplot(1,2,1)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Original")
    plt.axis('off')

    plt.subplot(1,2,2)
    plt.imshow(vessel_map, cmap='gray')
    plt.title("Zhang Vessel Map")
    plt.axis('off')

    plt.show()

    print(f"[{idx+1}/{len(image_paths)}] Saved: {save_path}")

print("All images processed and saved in:", output_folder)

In [ ]:
# A to Z Vessel Enhancement Pipeline for Fundus Images
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.filters import frangi
import os
import glob

# -------- 1. Set your dataset path --------
dataset_path = "/kaggle/input/datasets/amimulahasanrofik/abu-sayed/Fundus_CIMT_2903 Dataset/"
output_path = "/kaggle/working/vessel_maps/"
os.makedirs(output_path, exist_ok=True)

# -------- 2. Get list of images (50 sample) --------
image_paths = glob.glob(os.path.join(dataset_path, "*.png"))[:50]  # first 50 images

# -------- 3. Loop over images --------
for img_path in image_paths:
    # 3a. Load image
    img = cv2.imread(img_path)
    
    # 3b. Resize if needed
    img = cv2.resize(img, (512,512))
    
    # 3c. Extract green channel
    green = img[:,:,1]
    
    # 3d. Apply Zhang/Frangi vesselness filter
    vessel = frangi(green, scale_range=(1,5), scale_step=1)
    
    # 3e. Normalize to 0-255
    vessel = cv2.normalize(vessel, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    
    # 3f. Invert: vessels black, background white
    vessel = 255 - vessel
    
    # 3g. Apply CLAHE to enhance vessel contrast
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    vessel = clahe.apply(vessel)
    
    # 3h. Threshold to make vessels fully black
    _, vessel = cv2.threshold(vessel, 180, 255, cv2.THRESH_BINARY_INV)
    
    # -------- 4. Plot original vs vessel map --------
    fig, axes = plt.subplots(1,2, figsize=(10,5))
    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Original")
    axes[0].axis('off')
    
    axes[1].imshow(vessel, cmap='gray')
    axes[1].set_title("Vessel Enhanced")
    axes[1].axis('off')
    
    plt.show()
    
    # -------- 5. Save vessel map --------
    base_name = os.path.basename(img_path)
    save_path = os.path.join(output_path, f"vessel_{base_name}")
    cv2.imwrite(save_path, vessel)

print("✅ Vessel enhancement completed for 50 images!")